In [ ]:
import momepy
import osmnx as ox
import networkx as nx
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString, MultiLineString
import matplotlib.pyplot as plt
import h3
import folium
import random
import math
import alphashape
from rtree import index as rtree_index

In [ ]:
def import_graph(town, type_graph):
    network = ox.graph.graph_from_place(town, network_type = type_graph) # Импорт трпнспортной сети
    speed = ox.routing.add_edge_speeds(network)
    ox.routing.add_edge_travel_times(speed)
    G_nodes, G_edges = ox.graph_to_gdfs(network)

    fig, ax = plt.subplots(figsize=(10, 15))
    G_edges.plot(ax=ax)
    ax.tick_params(left = False, bottom = False, labelleft = False, labelbottom = False)
    ax.set_title('Транспортная сеть города Бронницы', fontsize=12)
    plt.show()
    return network, G_nodes, G_edges

def create_alpha_shape_for_roads(roads_gdf, alpha_value=0.1):
    # Объединение всех дорог в один MultiLineString
    all_roads = MultiLineString(roads_gdf.geometry.tolist())
    # Извлечение всех точек из линий дорог
    points = []
    for line in all_roads.geoms:
        if isinstance(line, LineString):
            points.extend(list(line.coords))
    # Построение альфа-оболочки по точкам
    alpha_shape = alphashape.alphashape(points, alpha_value)
    return alpha_shape

def gex_poly_roads_first(G_edges, resolution, centroid_size):
    #Создание оболочки для слоя дорог
    alpha_shape = create_alpha_shape_for_roads(G_edges, alpha_value=0.1)
    coords = list(alpha_shape.exterior.coords)
    swapped_coords = [(y, x) for x, y in coords]
    poly = h3.LatLngPoly(swapped_coords)

    #Построение гексакональной сетки, покрывающей исходный граф
    cells = h3.polygon_to_cells(poly, resolution)
    print(f"Number of H3 cells: {len(cells)}")
    cell_b = [h3.cell_to_boundary(i) for i in cells]

    cell_geoms = [h3.LatLngPoly(cell_id) for cell_id in cell_b]
    geo_df_cells = gpd.GeoDataFrame({'index': cells, 'geometry': cell_geoms}, crs='EPSG:4326')

    geo_df_cells_filtered=geo_df_cells[geo_df_cells['geometry'].intersects(alpha_shape)]

    centroids = [h3.cell_to_latlng(c) for c in geo_df_cells_filtered['index']]
    geometry = [Point(lng, lat) for lat, lng in centroids]
    df_cent = pd.DataFrame({"cells": geo_df_cells_filtered['index']})
    gdf_centroids = gpd.GeoDataFrame(df_cent, geometry=geometry, crs="EPSG:4326")

    fig, ax = plt.subplots(figsize=(10, 15))
    geo_df_cells_filtered.plot(ax=ax, color = 'lightblue', edgecolor = 'black', alpha = 0.5)
    gdf_centroids.plot(ax=ax, color='red', markersize=centroid_size)
    G_edges.plot(ax=ax)
    plt.show()
    return  geo_df_cells_filtered, gdf_centroids, centroids, alpha_shape

def normalize_pair(row): # Функция для нормализации пар
    return tuple(sorted([row['central_cell'], row['neighboring_cell']]))  # Сортируем и возвращаем кортеж

def cell_to_centroid(df, name): # Посик координат центроидов ячеек
    centr = []
    for cencel in df[name]:
        centr.append(h3.cell_to_latlng(cencel))
    return centr

def neighboring_gex(id_fill): # Поиск соседних ячеек
    neighboring_cells = []
    for h in id_fill:
        neighb = h3.grid_ring(h, k=1)
        neighboring_cells.append(neighb)

    # Список кортежей (центральная ячейка, соседняя ячейка)
    data = []
    for central, neighbors in zip(id_fill, neighboring_cells):
        for neighbor in neighbors:
            data.append((central, neighbor))

    # Сводная таблица
    tog_cells = pd.DataFrame(data, columns=['central_cell', 'neighboring_cell'])
    tog_cells['normalized_pair'] = tog_cells.apply(normalize_pair, axis=1)
    tog_cells = tog_cells.drop_duplicates(subset=['normalized_pair'], keep='first')
    tog_cells = tog_cells.drop(columns=['normalized_pair'])

    cencel = cell_to_centroid(tog_cells, 'central_cell')
    neicel = cell_to_centroid(tog_cells, 'neighboring_cell')

    tog_cells['geo_central_cell'] = cencel
    tog_cells['geo_neighboring_cell'] = neicel

    tog_cells = tog_cells.reset_index()
    return tog_cells

def filtered_cells(tog_cells, alpha_shape): # Отбор ячеек, лежащих внутри оболочки
    cell_b = [h3.cell_to_boundary(i) for i in tog_cells['neighboring_cell']]
    cell_geoms = [h3.LatLngPoly(cell_id2) for cell_id2 in cell_b]
    tog_cells2 = gpd.GeoDataFrame(tog_cells, geometry = cell_geoms, crs='EPSG:4326')
    tog_cells_filtered=tog_cells2[tog_cells2['geometry'].intersects(alpha_shape)]
    tog_cells_filtered = tog_cells_filtered.reset_index()
    return tog_cells_filtered

def create_edges(tog_cells_filtered): # Построение рёбер регулярно-сеточного графа
    lines = []
    for central2, neighbors2 in zip(tog_cells_filtered['geo_central_cell'], tog_cells_filtered['geo_neighboring_cell']):
        line = LineString([(central2[1], central2[0]), (neighbors2[1], neighbors2[0])])
        lines.append(line)

    gdf_lines = gpd.GeoDataFrame(geometry=lines)
    gdf_lines.set_crs(epsg=4326, inplace=True)
    return gdf_lines

def find_path(network, point1, point2): # Поиск кратчайших путей между всеми парами соседних центроидов
    try:
        origin_node = ox.distance.nearest_nodes(network, point1[1], point1[0])
        destination_node = ox.distance.nearest_nodes(network, point2[1], point2[0])

        # Нахождение кратчайшего пути по расстоянию
        shortest_path_by_distance = nx.shortest_path(network, source=origin_node, target=destination_node, weight='length')

        # Расчет расстояния
        distance = sum(ox.utils_graph.get_route_edge_attributes(network, shortest_path_by_distance, 'length'))
        return shortest_path_by_distance, distance

    except nx.NetworkXNoPath:
        print(f"Нет пути между {origin_node} и {destination_node}.")
        return None, None  # Возвращаем None, если путь не найден

def create_regular_graph(results_gdf): # Построение регулярно-сеточного графа
# Создание графа
    G_reg = momepy.gdf_to_nx(results_gdf, multigraph=False)
    G_nodes, G_edges = momepy.nx_to_gdf(G_reg)

    # Добавление пространственного индекса для ускорения вычислений
    spatial_idx = rtree_index.Index()
    geometry_dict = {}

    # Заполнение индекса и словаря
    for idx, row in results_gdf.iterrows():
        geom = row.geometry
        spatial_idx.insert(idx, geom.bounds)
        geometry_dict[idx] = (geom, row['distance'])

    # Добавление весов
    for u, v, data in G_reg.edges(data=True):
        edge_geom = data.get('geometry', None)
        if edge_geom is not None:
            for idx in spatial_idx.nearest(edge_geom.bounds, 1):
                cached_geom, dist = geometry_dict[idx]
                if cached_geom.equals(edge_geom):
                    data['weight'] = dist
                    break
    return G_reg, G_nodes, G_edges

def find_nearest_node(graph, target_coord):
    min_distance = float('inf')
    nearest_node = None
    for node in graph.nodes:
        node_coord = node
        distance = math.sqrt((node_coord[0] - target_coord[0])**2 + (node_coord[1] - target_coord[1])**2)
        if distance < min_distance:
            min_distance = distance
            nearest_node = node
    return nearest_node

def agregation(gdf, hex):
    joined = gpd.sjoin(gdf, hex, how='inner', predicate='within')
    result = joined.groupby('index')['value'].mean().reset_index()
    hex_with_avg = hex.merge(result, on='index', how='left')
    return hex_with_avg

In [ ]:
def main_function(network, G_edges_isx, resolution, dist, centroid_size):
    #Замощение исходного графа шестиугольной сеткой в границах выпуклой оболочки
    geo_df_cells_filtered, gdf_centroids, centroids, alpha_shape = gex_poly_roads_first(G_edges_isx, resolution, centroid_size)
    print('Замощение выполнено успешно')

    #Поиск соседних ячеек и удаление лишних
    id_fill = geo_df_cells_filtered['index']
    tog_cells = neighboring_gex(id_fill)
    tog_cells_filtered = filtered_cells(tog_cells, alpha_shape)
    tog_cells_filtered
    print('Поиск соседей выполнен успешно')

    #Создание ребер для регулярного графа
    gdf_lines = create_edges(tog_cells_filtered)

    fig, ax = plt.subplots(figsize=(10, 15))
    geo_df_cells_filtered.plot(ax=ax, color = 'lightblue', edgecolor = 'black', alpha = 0.5)
    G_edges_isx.plot(ax=ax)
    gdf_lines.plot(ax=ax, color = 'black', linewidth=0.5)
    gdf_centroids.plot(ax=ax, color='red', markersize=centroid_size, zorder = 2)
    plt.show()
    print('Ребра созданы')

    #Поиск маршрутов между соседними центроидами ячеек и рассчет их расстояний
    results = []
    for tog in range(len(tog_cells_filtered)):
        path, distance = find_path(network, tog_cells_filtered['geo_central_cell'][tog], tog_cells_filtered['geo_neighboring_cell'][tog])
        if path is not None:  # Если путь найден
            results.append({
                'central': tog_cells_filtered['geo_central_cell'][tog],
                'neighbor': tog_cells_filtered['geo_neighboring_cell'][tog],
                'path': path,
                'distance': distance
            })
        else:  # Если путь не найден
            results.append({
                'central': tog_cells_filtered['geo_central_cell'][tog],
                'neighbor': tog_cells_filtered['geo_neighboring_cell'][tog],
                'path': None,
                'distance': None
            })
            print(f"Нет пути между {tog_cells_filtered['geo_central_cell'][tog]} и {tog_cells_filtered['geo_neighboring_cell'][tog]}.")
    print(results)
    print('Веса ребер посчитаны')

    #Создание таблицы с идексами, координатами и расстоянием
    results_df = pd.DataFrame(results)
    results_gdf = gpd.GeoDataFrame(results_df, geometry=gdf_lines['geometry'])
    results_gdf.insert(0, 'index_central', tog_cells_filtered['central_cell'])
    results_gdf.insert(1, 'index_neighbor', tog_cells_filtered['neighboring_cell'])
    results_gdf['distance'] = results_gdf['distance'].replace(0, dist)
    results_gdf['distance'] = results_gdf['distance'].fillna(dist)
    results_gdf.to_crs(epsg=4326, inplace=True)
    print(results_gdf)
    print('Таблица создана')

    #Создание регулярного графа
    G_reg, G_nodes, G_edges = create_regular_graph(results_gdf)
    fig, ax = plt.subplots(figsize=(10, 15))
    G_edges.plot(ax=ax)
    ax.tick_params(left = False, bottom = False, labelleft = False, labelbottom = False)
    plt.show()
    print('Регулярный граф создан!!!!!')
    return centroids, G_reg, G_nodes, G_edges, geo_df_cells_filtered

### Входные данные для проведения вычислений

In [ ]:
#Ввод названия города и типа графа
town = "Bronnitsy, Russia"
type_graph = "walk"

#Скачивание транспортной сети с использованием библиотеки osmnx
network, G_nodes_isx, G_edges_isx = import_graph(town, type_graph)
print('Данные скачаны')

### Преобразование в регулярно-сеточный вид

In [ ]:
#Разрешение шестиугольной сетки и расстояние по умолчанию
resolution_9 = 9
dist_9 = 337
centroid_size_9 = 50

centroids_9, G_reg_9, G_nodes_9, G_edges_9, geo_df_cells_filtered_9 = main_function(network, G_edges_isx, resolution_9, dist_9, centroid_size_9)

In [ ]:
resolution_10 = 10
dist_10 = 127
centroid_size_10 = 8

centroids_10, G_reg_10, G_nodes_10, G_edges_10, geo_df_cells_filtered_10 = main_function(network, G_edges_isx, resolution_10, dist_10, centroid_size_10)

In [ ]:
resolution_11 = 11
dist_11 = 47
centroid_size_11 = 1

centroids_11, G_reg_11, G_nodes_11, G_edges_11, geo_df_cells_filtered_11 = main_function(network, G_edges_isx, resolution_11, dist_11, centroid_size_11)

### Коэффициент корреляции и детерминации

In [ ]:
def time_for_corr(G_reg, centroids):
    dist_res = pd.DataFrame(columns=['Point1', 'Point2', 'Original_Path_Length', 'Regular_Path_Length'])

    # Выполняем поиск 400 расстояний
    for _ in range(400):
        point1 = random.choice(list(centroids))
        point2 = random.choice(list(centroids))

        # Находим ближайшие узлы
        start_node = find_nearest_node(G_reg, (point1[1], point1[0]))
        end_node = find_nearest_node(G_reg, (point2[1], point2[0]))

        # Проверяем, что узлы найдены
        if start_node is None or end_node is None:
            raise ValueError("Не удалось найти ближайшие узлы в графе.")

        # Находим кратчайший путь
        shortest_path = nx.shortest_path(G_reg, source=start_node, target=end_node, weight='weight')
        shortest_path_length = nx.shortest_path_length(G_reg, source=start_node, target=end_node, weight='weight')

        # Предположим, что у вас есть функция find_path, которая возвращает путь и расстояние
        path, distance = find_path(network, point1, point2)

        # Добавляем результаты в DataFrame
        new_row = pd.DataFrame({
            'Point1': [point1],
            'Point2': [point2],
            'Original_Path_Length': [distance],
            'Regular_Path_Length': [shortest_path_length]
        })

        # Добавляем новую строку в DataFrame с помощью pd.concat
        dist_res = pd.concat([dist_res, new_row], ignore_index=True)
        dist_res['Original_time'] = dist_res['Original_Path_Length'] / (5*1000/60)
        dist_res['Regular_time'] = dist_res['Regular_Path_Length'] / (5*1000/60)
    return dist_res

In [ ]:
dist_res_9 = time_for_corr(G_reg_9, centroids_9)
dist_res_10 = time_for_corr(G_reg_10, centroids_10)
dist_res_11 = time_for_corr(G_reg_11, centroids_11)

In [ ]:
import seaborn as sns
from sklearn.metrics import r2_score
import scipy.stats
# Настройка стиля
sns.set(style="whitegrid")

# Создаем фигуру с тремя subplot'ами
fig, axes = plt.subplots(1, 3, figsize=(18, 6),
                       gridspec_kw={'wspace': 0.05})  # Минимальное расстояние между графиками

# Данные (замените на ваши DataFrame)
datasets = [dist_res_9, dist_res_10, dist_res_11]  # Пример: три набора данных
titles = ["Уровень 9", "Уровень 10", "Уровень 11"]  # Заголовки

# Находим общие пределы для всех графиков
x_min = min(df['Original_time'].min() for df in datasets)
x_max = max(df['Original_time'].max() for df in datasets)
y_min = min(df['Regular_time'].min() for df in datasets)
y_max = max(df['Regular_time'].max() for df in datasets)

# Добавляем небольшой отступ (5% от диапазона)
x_pad = (x_max - x_min) * 0.05
y_pad = (y_max - y_min) * 0.05

x_limits = (x_min - x_pad, x_max + x_pad)
y_limits = (y_min - y_pad, y_max + y_pad)

# Построение графиков
for i, (ax, data, title) in enumerate(zip(axes, datasets, titles)):
    correlation = data['Original_time'].corr(data['Regular_time'])
    r2 = correlation*correlation
    #Критерий значимости
    r, p = scipy.stats.pearsonr(data['Original_time'], data['Regular_time'])
    print(r)
    # Scatterplot
    sns.scatterplot(x='Original_time', y='Regular_time', data=data, alpha=0.7, edgecolor='k', ax=ax)
    # Линия тренда
    sns.regplot(x='Original_time', y='Regular_time', data=data, scatter=False, color='red', line_kws={'linestyle': '--'}, ax=ax)
    # Настройка осей
    ax.set_xlim(x_limits)
    ax.set_ylim(y_limits)
    ax.set_aspect('equal', adjustable='box')
    # Линии сетки через ноль
    ax.axhline(0, color='gray', linestyle='-', linewidth=0.5)
    ax.axvline(0, color='gray', linestyle='-', linewidth=0.5)
    # Подписи только для первого графика (чтобы избежать дублирования)
    if i == 0:
        ax.set_ylabel('Время по регулярному графу, мин', fontsize=10)
    else:
        ax.set_ylabel('')

    ax.set_xlabel('Время по исходному графу, мин', fontsize=10)
    ax.set_title(f'{title}\nКоэффициент корреляции: {correlation:.3f}\nКоэффициент детерминации: {r2:.3f}', fontsize=12)
    ax.grid(True)
    print(r2)

# Общий заголовок
plt.suptitle("Зависимость между затраченным временем по исходному и регулярному графам", fontsize=14, y=1.02)

# Убираем лишние отступы
plt.tight_layout(pad=1.0)  # Минимальный отступ вокруг всей фигуры
plt.show()

### Построение зон доступности

In [ ]:
#Поиск кратчайших расстояний для регулярного графа
point1 = (55.6995, 37.828)
start_node = find_nearest_node(G_reg_9, (point1[1], point1[0]))
path_lengths = nx.single_source_dijkstra_path_length(G_reg_9, start_node, cutoff=None, weight='weight')

features = []
for coords, value in path_lengths.items():
    lon, lat = coords
    point = Point(lon, lat)
    features.append({"geometry": point, "value": value})
gdf10 = gpd.GeoDataFrame(features)
gdf10.set_crs(epsg=4326, inplace=True)

#Центральная вершина
r = [{"geometry": Point(point1[1], point1[0]), "value": 11111}]
gdf11 = gpd.GeoDataFrame(r)
gdf11.set_crs(epsg=4326, inplace=True)

#Поиск кратчайших расстояний для исходного графа
start_node = ox.distance.nearest_nodes(network, point1[1], point1[0])
path_lengths2 = nx.single_source_dijkstra_path_length(network, start_node, cutoff=None, weight='length')

features2 = []
for index, value in path_lengths2.items():
    features2.append({"osmid": index, "value": value})

geometry_lookup = {osmid: row["geometry"] for osmid, row in G_nodes_isx.iterrows()}
for item in features2:
    item_id = item["osmid"]
    if item_id in geometry_lookup:
        item["geometry"] = geometry_lookup[item_id]

gdf12 = gpd.GeoDataFrame(features2)
gdf12.set_crs(epsg=4326, inplace=True)
print()

#Экспорт вершин с записанным значением удаленности от центральной точки
gdf10.to_file("izo_reg11.geojson", driver="GeoJSON")
gdf11.to_file("izo_glav11.geojson", driver="GeoJSON")
gdf12.to_file("izo_geo11.geojson", driver="GeoJSON")

#Агрегация данных на шестиугольной сетки построение изохрон
hex_reg = geo_df_cells_filtered_9.copy()
hex_geo = geo_df_cells_filtered_9.copy()
hex_with_avg_reg = agregation(gdf10, hex_reg)
hex_with_avg_geo = agregation(gdf12, hex_geo)

hex_with_avg_reg.to_file("izo_reg_hex11.geojson", driver="GeoJSON")
hex_with_avg_geo.to_file("izo_geo_hex11.geojson", driver="GeoJSON")

### Подсчет количества вершин и ребер

In [ ]:
def count_vertices_and_edges(graph):
    vertices = len(graph.nodes())
    edges = len(graph.edges())
    avg_degree = round(2 * edges / vertices, 2) if vertices > 0 else 0  # Средняя степень вершины
    density = round(2*edges / (vertices * (vertices - 1)), 4) if vertices > 1 else 0  # Плотность
    return vertices, edges, avg_degree, density

# Словарь с графами (замените на ваши реальные графы)
graphs = {
    "Исходный граф": network,
    "Регулярный граф (9 уровень)": G_reg_9,
    "Регулярный граф (10 уровень)": G_reg_10,
    "Регулярный граф (11 уровень)": G_reg_11
}

# Сбор данных
results = []
for name, graph in graphs.items():
    v, e, avg_deg, dens = count_vertices_and_edges(graph)
    results.append({
        "Граф": name,
        "Вершины (V)": v,
        "Рёбра (E)": e,
        "E/V": round(e/v, 2) if v != 0 else 0,
        "Ср. степень": avg_deg,
        "Плотность": dens
    })

# Создание и вывод таблицы
df = pd.DataFrame(results)
print("\n" + df.to_string(index=False, justify='center'))

### Расчет центральностей

In [ ]:
def harm_cent(graph):
    hc = nx.harmonic_centrality(graph)
    nx.set_node_attributes(graph, hc, "HC")
    G_nodes, G_edges = momepy.nx_to_gdf(graph)
    return G_nodes, G_edges

def clos_cent(graph):
    bc = nx.closeness_centrality(graph)
    nx.set_node_attributes(graph, bc, "BC")
    G_nodes, G_edges = momepy.nx_to_gdf(graph)
    return G_nodes, G_edges

def betc_cent(graph):
    betc = nx.betweenness_centrality(graph)
    nx.set_node_attributes(graph, betc, "betc")
    G_nodes, G_edges = momepy.nx_to_gdf(graph)
    return G_nodes, G_edges

In [ ]:
G_nodes_reg_bc, G_edges_reg_bc = clos_cent(G_reg_9)
map = folium.Map(location=[55.421505, 38.259344], zoom_start=14)
G_edges_reg_bc.explore(m = map, color = 'grey', weight=1)
G_nodes_reg_bc.explore(m = map,
                column="BC",
                scheme="quantiles",
                legend=True,
                k=5,
                legend_kwds=dict(colorbar=False, caption= r"Центральность по близости"),
                cmap="Reds",
                marker_kwds=dict(radius=2, fill=True))
map

In [ ]:
G_nodes_geo_bc, G_edges_geo_bc = clos_cent(network)
map_geo = folium.Map(location=[55.421505, 38.259344], zoom_start=14)
G_edges_geo_bc.explore(m = map_geo, color = 'grey', weight=1)
G_nodes_geo_bc.explore(m = map_geo,
                column="BC",
                scheme="quantiles",
                legend=True,
                k=5,
                legend_kwds=dict(colorbar=False, caption= r"Центральность по близости"),
                cmap="Reds",
                marker_kwds=dict(radius=1, fill=True))
map_geo

In [ ]:
G_nodes_reg_betc, G_edges_reg_betc = betc_cent(G_reg_9)
map = folium.Map(location=[55.421505, 38.259344], zoom_start=14)
G_edges_reg_betc.explore(m = map, color = 'grey', weight=1)
G_nodes_reg_betc.explore(m = map,
                column="betc",
                scheme="quantiles",
                legend=True,
                k=5,
                legend_kwds=dict(colorbar=False, caption= r"Центральность по промежуточности"),
                cmap="Reds",
                marker_kwds=dict(radius=2, fill=True))
map

In [ ]:
G_nodes_geo_betc, G_edges_geo_betc = betc_cent(network)
map_geo = folium.Map(location=[55.421505, 38.259344], zoom_start=14)
G_edges_geo_betc.explore(m = map_geo, color = 'grey', weight=1)
G_nodes_geo_betc.explore(m = map_geo,
                column="betc",
                scheme="quantiles",
                legend=True,
                k=5,
                legend_kwds=dict(colorbar=False, caption= r"Центральность по промежуточности"),
                cmap="Reds",
                marker_kwds=dict(radius=1, fill=True))
map_geo

In [ ]:
betc_line = nx.edge_betweenness_centrality(G_reg_9)
nx.set_edge_attributes(G_reg_9, betc_line, "betc_line")
G_nodes_reg_betc_line, G_edges_reg_betc_line = momepy.nx_to_gdf(G_reg_9)

In [ ]:
map = folium.Map(location=[55.421505, 38.259344], zoom_start=14)
for _, row in G_edges_reg_betc_line.iterrows():
    bc_value = row['betc_line']
    # Пропорционально масштабируем толщину линии (минимум 1, максимум 10)
    weight = 1 + (bc_value - G_edges_reg_betc_line['betc_line'].min()) / (G_edges_reg_betc_line['betc_line'].max() - G_edges_reg_betc_line['betc_line'].min()) * 9
    # Добавляем линию на карту с нужной толщиной
    folium.PolyLine(
        locations=[(pt[1], pt[0]) for pt in row['geometry'].coords],
        weight=weight,
        color='red',
        opacity=0.7
    ).add_to(map)
map

In [ ]:
betc_line = nx.edge_betweenness_centrality(G_reg_10)
nx.set_edge_attributes(G_reg_10, betc_line, "betc_line")
G_nodes_reg_betc_line, G_edges_reg_betc_line = momepy.nx_to_gdf(G_reg_10)

map = folium.Map(location=[55.421505, 38.259344], zoom_start=14)
for _, row in G_edges_reg_betc_line.iterrows():
    bc_value = row['betc_line']
    # Пропорционально масштабируем толщину линии (минимум 1, максимум 10)
    weight = 1 + (bc_value - G_edges_reg_betc_line['betc_line'].min()) / (G_edges_reg_betc_line['betc_line'].max() - G_edges_reg_betc_line['betc_line'].min()) * 9
    # Добавляем линию на карту с нужной толщиной
    folium.PolyLine(
        locations=[(pt[1], pt[0]) for pt in row['geometry'].coords],
        weight=weight,
        color='red',
        opacity=0.7
    ).add_to(map)
map

In [ ]:
betc_line = nx.edge_betweenness_centrality(network)
nx.set_edge_attributes(network, betc_line, "betc_line")
G_nodes_geo_betc_line, G_edges_geo_betc_line = ox.graph_to_gdfs(network)

In [ ]:
map_geo = folium.Map(location=[55.421505, 38.259344], zoom_start=14)
for _, row in G_edges_geo_betc_line.iterrows():
    bc_value = row['betc_line']
    # Пропорционально масштабируем толщину линии (минимум 1, максимум 10)
    weight = 1 + (bc_value - G_edges_geo_betc_line['betc_line'].min()) / (G_edges_geo_betc_line['betc_line'].max() - G_edges_geo_betc_line['betc_line'].min()) * 9
    # Добавляем линию на карту с нужной толщиной
    folium.PolyLine(
        locations=[(pt[1], pt[0]) for pt in row['geometry'].coords],
        weight=weight,
        color='red',
        opacity=0.7
    ).add_to(map_geo)
map_geo